# Coadd multi-frequency sky maps (100 / 143 / 353 GHz)

Follows the **next step** in the component notebooks:

1. `simulate_lensed_primary_cmb.ipynb` → lensed primary CMB
2. `simulate_tsz_frequency_maps.ipynb` → $\Delta T_\mathrm{tSZ}(\nu)$
3. `simulate_cib_frequency_maps.ipynb` → $\Delta T_\mathrm{CIB}(\nu)$
   (353 GHz = released map; 100/143 GHz = notebook SED-scaling from 217 GHz)

Then coadd

$$
T_\nu = T_\mathrm{CMB} + \Delta T_\mathrm{tSZ}(\nu) + \Delta T_\mathrm{CIB}(\nu)
\quad[\mu\mathrm{K}_\mathrm{CMB}].
$$

**Outputs** (kept for post-compsep visualization):

| Kind | Path |
|------|------|
| Raw components | `maps_100_143_353/raw/` |
| Coadded skies | `maps_100_143_353/coadd/` |
| ILC products | `maps_100_143_353/ilc_output/` |

> **Kernel**: `cosmo_env`. $N_\mathrm{side}=4096$.


In [ ]:
from pathlib import Path
import os

import h5py
import healpy as hp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from astropy import constants as const
from astropy import units as u

DATA_DIR = Path(
    "/home/ext_andyxlcnb_gmail_com/cosmology_data/flamingo"
    "/L2p8_m9/integrated_maps/yang26/lightcone0_shells"
)
SIM_DIR = DATA_DIR / "simulated"
MAP_ROOT = Path(
    "/home/ext_andyxlcnb_gmail_com/flamingo_mock_analysis/maps_100_143_353"
)
RAW_DIR = MAP_ROOT / "raw"
COADD_DIR = MAP_ROOT / "coadd"

NSIDE = 4096
T_CMB = 2.7255
SEED = 42
FREQUENCIES_GHZ = [100.0, 143.0, 353.0]

# CIB SED (Yang et al. 2026 three-parameter model; same as simulate_cib_frequency_maps)
BETA_D = 1.65
T0 = 35.14
ALPHA = 0.0
Z_EFF = 1.5
RELEASED_CIB = {
    217: "lensed_CIB_rot_BANDPASS_F217_three_params_same_rot.hdf5",
    353: "lensed_CIB_rot_BANDPASS_F353_three_params_same_rot.hdf5",
}

RAW_DIR.mkdir(parents=True, exist_ok=True)
COADD_DIR.mkdir(parents=True, exist_ok=True)
print(f"Nside={NSIDE}, freqs={FREQUENCIES_GHZ}")
print(f"raw  -> {RAW_DIR}")
print(f"coadd-> {COADD_DIR}")


## 1. Spectral helpers (from component notebooks)

In [ ]:
def tsz_f(nu_ghz: float, t_cmb: float = T_CMB) -> float:
    x = (const.h * nu_ghz * u.GHz / (const.k_B * t_cmb * u.K)).to_value(
        u.dimensionless_unscaled
    )
    return float(x / np.tanh(0.5 * x) - 4.0)


def y_to_delta_T_uK(y: np.ndarray, nu_ghz: float) -> np.ndarray:
    return (T_CMB * 1.0e6) * y * tsz_f(nu_ghz)


def dB_dT_Jy_per_sr_per_K(nu_ghz: float, t_cmb: float = T_CMB) -> float:
    nu = nu_ghz * u.GHz
    T = t_cmb * u.K
    x = (const.h * nu / (const.k_B * T)).to_value(u.dimensionless_unscaled)
    prefactor_si = (2.0 * const.k_B * nu**2 / const.c**2).to_value(
        u.W / u.m**2 / u.Hz / u.K
    )
    dBdT_si = prefactor_si * (x**2 * np.exp(x) / np.expm1(x) ** 2)
    return dBdT_si / 1e-26


def intensity_to_uK(I_Jy_sr: np.ndarray, nu_ghz: float) -> np.ndarray:
    return I_Jy_sr / dB_dT_Jy_per_sr_per_K(nu_ghz) * 1.0e6


def theta_nu(nu_ghz, t_dust, beta_d=BETA_D):
    nu = np.asarray(nu_ghz, dtype=np.float64)
    x = (const.h * nu * u.GHz / (const.k_B * t_dust * u.K)).to_value(
        u.dimensionless_unscaled
    )
    x = np.clip(x, 1e-8, 100.0)
    return (nu ** (beta_d + 3.0)) / np.expm1(x)


def sed_shape_observed(nu_obs_ghz, z):
    t = T0 * (1.0 + z) ** ALPHA
    return theta_nu(np.asarray(nu_obs_ghz, dtype=np.float64) * (1.0 + z), t)


def load_hdf5(path: Path) -> np.ndarray:
    with h5py.File(path, "r") as f:
        m = f["data"][:].astype(np.float64)
    if m.size != 12 * NSIDE**2:
        raise ValueError(f"{path.name}: unexpected size {m.size}")
    return m


def write_map(path: Path, m, unit: str, freq=None, extra=None, dtype=np.float32):
    hdr = [("UNIT", unit), ("NSIDE", int(NSIDE))]
    if freq is not None:
        hdr.append(("FREQ", float(freq), "GHz"))
    if extra:
        hdr.extend(extra)
    hp.write_map(
        path,
        np.asarray(m, dtype=dtype),
        overwrite=True,
        dtype=dtype,
        column_names=["TEMPERATURE"] if "K" in unit or unit == "uK_CMB" else ["DATA"],
        extra_header=hdr,
    )
    print(f"  wrote {path} ({path.stat().st_size / 1e9:.2f} GB)")

## 2. Raw maps: CMB, Compton-$y$, tSZ($\nu$), CIB($\nu$)

In [ ]:
# Lensed CMB (from simulate_lensed_primary_cmb; already cached at Nside=4096)
cmb_src = SIM_DIR / f"primary_CMB_T_lensed_nside{NSIDE}_seed{SEED}.fits"
assert cmb_src.is_file(), f"Missing CMB — run simulate_lensed_primary_cmb.ipynb first: {cmb_src}"
cmb_link = RAW_DIR / cmb_src.name
if cmb_link.is_symlink() or cmb_link.exists():
    cmb_link.unlink()
os.symlink(cmb_src, cmb_link)
cmb_uK = hp.read_map(str(cmb_src), dtype=np.float64)
print(f"CMB: std={cmb_uK.std():.2f} uK  (symlink {cmb_link} -> {cmb_src})")

# True Compton-y
print("Loading lensed y...")
y = load_hdf5(DATA_DIR / "lensed_tSZ_rot_same_rot.hdf5")
write_map(
    RAW_DIR / f"compton_y_nside{NSIDE}.fits",
    y,
    unit="Compton_y",
    extra=[("COMP", "tSZ_y")],
    dtype=np.float32,
)

# tSZ ΔT at 100/143/353 (simulate_tsz_frequency_maps)
print("Building tSZ frequency maps...")
tsz_uK = {}
for nu in FREQUENCIES_GHZ:
    dt = y_to_delta_T_uK(y, nu)
    tsz_uK[nu] = dt
    write_map(
        RAW_DIR / f"tSZ_deltaT_{nu:.0f}GHz_nside{NSIDE}.fits",
        dt,
        unit="uK_CMB",
        freq=nu,
        extra=[("COMP", "tSZ"), ("KERNEL", "nonrel_f(x)")],
    )
    print(f"  tSZ {nu:g} GHz: std={dt.std():.3f} uK")


In [ ]:
# CIB: 353 = released; 100/143 = SED scale from 217 (simulate_cib_frequency_maps)
print("Loading released CIB intensity (217, 353)...")
cib_I = {nu: load_hdf5(DATA_DIR / fname) for nu, fname in RELEASED_CIB.items()}


def approximate_cib_intensity(nu_ghz: float):
    bands = np.array(sorted(cib_I), dtype=float)
    for b in bands:
        if abs(nu_ghz - b) < 0.5:
            return cib_I[int(b)].copy(), f"released {int(b)} GHz"
    nu_ref = float(bands[np.argmin(np.abs(bands - nu_ghz))])
    ratio = float(
        sed_shape_observed(nu_ghz, Z_EFF) / sed_shape_observed(nu_ref, Z_EFF)
    )
    return (
        cib_I[int(nu_ref)] * ratio,
        f"SED scale from {nu_ref:.0f} GHz at z_eff={Z_EFF} (x{ratio:.3f})",
    )


print("Building CIB frequency maps...")
cib_uK = {}
cib_methods = {}
for nu in FREQUENCIES_GHZ:
    I, method = approximate_cib_intensity(nu)
    T = intensity_to_uK(I, nu)
    del I
    cib_uK[nu] = T
    cib_methods[nu] = method
    approx = int("released" not in method)
    write_map(
        RAW_DIR / f"CIB_deltaT_{nu:.0f}GHz_nside{NSIDE}.fits",
        T,
        unit="uK_CMB",
        freq=nu,
        extra=[
            ("COMP", "CIB"),
            ("APPROX", approx),
            ("METHOD", method[:60].encode("ascii", "replace").decode("ascii")),
        ],
    )
    print(f"  CIB {nu:g} GHz: std={T.std():.3f} uK | {method}")

del cib_I


## 3. Coadded skies $T_\nu = \mathrm{CMB} + \mathrm{tSZ} + \mathrm{CIB}$

Save both $\mu\mathrm{K}$ (visualization / bookkeeping) and $\mathrm{K}_\mathrm{CMB}$ (pyILC input).

In [ ]:
print("Coadding...")
for nu in FREQUENCIES_GHZ:
    total_uK = cmb_uK + tsz_uK[nu] + cib_uK[nu]
    print(
        f"  {nu:g} GHz: CMB={cmb_uK.std():.2f} | tSZ={tsz_uK[nu].std():.2f} | "
        f"CIB={cib_uK[nu].std():.2f} | total={total_uK.std():.2f} uK"
    )
    write_map(
        COADD_DIR / f"sky_CMB_tSZ_CIB_{nu:.0f}GHz_nside{NSIDE}_uK.fits",
        total_uK,
        unit="uK_CMB",
        freq=nu,
        extra=[("COMPS", "CMB+tSZ+CIB")],
        dtype=np.float32,
    )
    write_map(
        COADD_DIR / f"sky_CMB_tSZ_CIB_{nu:.0f}GHz_nside{NSIDE}_K.fits",
        total_uK * 1.0e-6,
        unit="K_CMB",
        freq=nu,
        extra=[("COMPS", "CMB+tSZ+CIB")],
        dtype=np.float64,
    )
    del total_uK

print("\nRaw:")
for p in sorted(RAW_DIR.iterdir()):
    print(f"  {p.name}")
print("Coadd:")
for p in sorted(COADD_DIR.iterdir()):
    print(f"  {p.name}")

## 4. Quick-look gallery (raw vs coadd)

In [ ]:
mpl.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200})

fig = plt.figure(figsize=(12, 10))
for i, nu in enumerate(FREQUENCIES_GHZ):
    for j, (m, title) in enumerate(
        [
            (tsz_uK[nu], rf"tSZ ${nu:.0f}$ GHz"),
            (cib_uK[nu], rf"CIB ${nu:.0f}$ GHz"),
            (cmb_uK + tsz_uK[nu] + cib_uK[nu], rf"coadd ${nu:.0f}$ GHz"),
        ]
    ):
        idx = i * 3 + j + 1
        vmax = np.percentile(np.abs(m), 99)
        hp.mollview(
            m,
            title=title,
            unit=r"$\mu\mathrm{K}$",
            min=-vmax if j != 1 else np.percentile(m, 1),
            max=vmax if j != 1 else np.percentile(m, 99),
            sub=(3, 3, idx),
            hold=True,
        )
plt.suptitle(r"Raw components and coadds (Nside=4096)", y=1.02)
fig = plt.gcf()
fig.savefig(MAP_ROOT / "raw_and_coadd_gallery.png", bbox_inches="tight")
plt.show()